# Mean Squared Error (MSE) — Beginner Masterclass

**Reading:** [GeeksforGeeks — Mean Squared Error](https://www.geeksforgeeks.org/maths/mean-squared-error/) and [Loss Functions in Deep Learning](https://www.geeksforgeeks.org/deep-learning/loss-functions-in-deep-learning/)

This Colab-compatible notebook explains MSE with manual arithmetic, plain Python, NumPy, PyTorch, TensorFlow/Keras, visualizations, and a small training experiment.

## Learning goals
- Calculate residuals and squared errors by hand.
- Understand why MSE penalizes large errors strongly.
- Compare a pure-Python implementation with library implementations.
- Use MSE to train a small regression model.
- Learn common implementation mistakes and where to check the official docs.

## 1. Formula and intuition

MSE is most commonly used for **regression**, where the target is a continuous value such as a price or temperature.

\[
\boxed{\operatorname{MSE}=\frac{1}{n}\sum_{i=1}^{n}(y_i-\hat y_i)^2}
\]

\(y_i\) is the actual target, \(\hat y_i\) is the prediction, and \(n\) is the number of values. The residual is \(e_i=y_i-\hat y_i\). Square each residual, sum the squares, and divide by \(n\).

MSE is sensitive to outliers because the penalty grows quadratically. Its units are the target units squared. RMSE \(=\sqrt{\operatorname{MSE}}\) is in the target's original units.

## 2. Worked example from the GeeksforGeeks article

Actual values: \([10,20,30,40,50]\)

Predictions: \([12,18,32,38,48]\)

Residuals: \([-2,2,-2,2,2]\)

Squared residuals: \([4,4,4,4,4]\)

\[
\operatorname{MSE}=\frac{4+4+4+4+4}{5}=\boxed{4}
\]

In [ ]:
# Pure Python: no numerical libraries used for this calculation
y_true = [10, 20, 30, 40, 50]
y_pred = [12, 18, 32, 38, 48]
errors = [a - p for a, p in zip(y_true, y_pred)]
squared_errors = [e ** 2 for e in errors]
mse = sum(squared_errors) / len(squared_errors)
print("Errors:", errors)
print("Squared errors:", squared_errors)
print("Sum of squares:", sum(squared_errors))
print("Number of values:", len(squared_errors))
print("MSE:", mse)
print("RMSE:", mse ** 0.5)

## 3. Same calculation with NumPy

In [ ]:
import numpy as np
y_np = np.array(y_true, dtype=np.float64)
p_np = np.array(y_pred, dtype=np.float64)
squared = (y_np - p_np) ** 2
mse_numpy = np.mean(squared)
print("Squared errors:", squared)
print("MSE:", mse_numpy)
assert np.isclose(mse_numpy, 4.0)

## 4. Why are large errors expensive?

In [ ]:
import matplotlib.pyplot as plt
residual = np.linspace(-10, 10, 401)
plt.figure(figsize=(8, 4))
plt.plot(residual, residual ** 2)
plt.title("Squared error grows quadratically")
plt.xlabel("Residual (actual - prediction)")
plt.ylabel("Squared error")
plt.grid(alpha=.25)
plt.show()
for e in [1, 2, 5, 10]:
    print(f"Residual {e}: squared error = {e**2}")

## 5. PyTorch MSELoss

`torch.nn.MSELoss()` calculates the elementwise squared differences and, by default, averages them. The target and predictions should have matching shapes for the intended comparison.

**Common gotcha:** prediction shaped `(N, 1)` and target shaped `(N,)` can broadcast into an unintended `(N, N)` comparison. Explicitly match the shapes.

In [ ]:
import torch
import torch.nn as nn
yt = torch.tensor(y_true, dtype=torch.float32)
yp = torch.tensor(y_pred, dtype=torch.float32)
print("MSELoss mean:", nn.MSELoss(reduction="mean")(yp, yt).item())
print("Elementwise losses:", nn.MSELoss(reduction="none")(yp, yt))
print("MSELoss sum:", nn.MSELoss(reduction="sum")(yp, yt).item())

## 6. TensorFlow / Keras MSE

In [ ]:
import tensorflow as tf
yt_tf = tf.constant(y_true, dtype=tf.float32)
yp_tf = tf.constant(y_pred, dtype=tf.float32)
loss_tf = tf.keras.losses.MeanSquaredError()(yt_tf, yp_tf)
print("Keras MSE:", float(loss_tf.numpy()))
assert np.isclose(float(loss_tf.numpy()), 4.0, atol=1e-6)

## 7. Derivative and training

For \(L=\frac1n\sum_i(\hat y_i-y_i)^2\),

\[
\boxed{\frac{\partial L}{\partial \hat y_i}=\frac{2}{n}(\hat y_i-y_i)}
\]

This tells gradient descent how each prediction should move to reduce the loss.

In [ ]:
pred = torch.tensor([12., 18., 32., 38., 48.], requires_grad=True)
target = torch.tensor([10., 20., 30., 40., 50.])
loss = nn.MSELoss()(pred, target)
loss.backward()
print("Autograd gradient:", pred.grad.numpy())
print("Analytical gradient:", (2 / len(y_true)) * (np.array(y_pred) - np.array(y_true)))

## 8. Train a small model on generated data

The data are generated in this notebook, so no upload is needed. The target follows approximately \(y=2.5x-0.7\), with added noise. A one-layer linear model learns its slope and intercept by minimizing MSE.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
rng = np.random.default_rng(7)
torch.manual_seed(7)
X = rng.uniform(-2, 2, (800, 1)).astype("float32")
Y = (2.5 * X - .7 + rng.normal(0, .3, (800, 1))).astype("float32")
loader = DataLoader(TensorDataset(torch.from_numpy(X), torch.from_numpy(Y)),
                    batch_size=32, shuffle=True)
model = nn.Linear(1, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=.03)
criterion = nn.MSELoss()
losses = []
for epoch in range(50):
    total, count = 0., 0
    for xb, yb in loader:
        optimizer.zero_grad()
        pred = model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()
        total += loss.item() * len(xb)
        count += len(xb)
    losses.append(total / count)
print("Learned slope:", model.weight.item())
print("Learned intercept:", model.bias.item())
print("Final MSE:", losses[-1])
plt.figure(figsize=(7,4)); plt.plot(losses)
plt.title("Training MSE"); plt.xlabel("Epoch"); plt.ylabel("MSE"); plt.grid(alpha=.25); plt.show()
with torch.no_grad():
    order = np.argsort(X[:,0])
    fitted = model(torch.from_numpy(X)).numpy()
plt.figure(figsize=(7,4))
plt.scatter(X[:,0], Y[:,0], s=10, alpha=.3, label="Data")
plt.plot(X[order,0], fitted[order,0], color="red", label="Fitted line")
plt.legend(); plt.title("Regression fit using MSE"); plt.grid(alpha=.2); plt.show()

## 9. Common mistakes and quick quiz

- MSE is usually a regression loss, not the default loss for ordinary classification.
- The default `"mean"` differs from `"sum"` and `"none"`.
- MSE is sensitive to outliers; compare it with MAE or Huber when appropriate.
- Make prediction and target shapes match.
- Targets/predictions should normally use compatible floating-point types.
- MSE has squared units; RMSE has original target units.

**Quiz:** Actual `[2,4,6]`, predictions `[1,5,4]`. What is MSE? Answer: \((1+1+4)/3=2\).

## Sources and official documentation

### All six reading links supplied for this series
- [GeeksforGeeks — Loss Functions in Deep Learning](https://www.geeksforgeeks.org/deep-learning/loss-functions-in-deep-learning/)
- [GeeksforGeeks — Mean Squared Error](https://www.geeksforgeeks.org/maths/mean-squared-error/)
- [GeeksforGeeks — MAE in PyTorch](https://www.geeksforgeeks.org/python/how-to-measure-the-mean-absolute-error-mae-in-pytorch/)
- [Medium — Mean Absolute Error](https://medium.com/@20__80__/mean-absolute-error-mae-machine-learning-ml-b9b4afc63077)
- [GeeksforGeeks — Binary Cross-Entropy](https://www.geeksforgeeks.org/deep-learning/binary-cross-entropy-log-loss-for-binary-classification/)
- [GeeksforGeeks — Categorical Cross-Entropy](https://www.geeksforgeeks.org/deep-learning/categorical-cross-entropy-in-multi-class-classification/)

### API documentation useful for debugging
- [PyTorch MSELoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.MSELoss.html)
- [Keras regression losses (MSE and MAE)](https://keras.io/api/losses/regression_losses/)
- [NumPy mean](https://numpy.org/doc/stable/reference/generated/numpy.mean.html)
- [PyTorch autograd tutorial](https://docs.pytorch.org/tutorials/beginner/basics/autogradqs_tutorial.html)

When debugging, check the full error message, tensor shapes, dtypes, reduction setting, and the official API docs. This notebook paraphrases the sources rather than reproducing them verbatim.